# Clase 1 - Complemento: Distribuciones Adicionales

**Tópicos de Estadística Avanzada**  
Dra. Beatriz Marrón - Dr. José Bavio  
Universidad Nacional del Sur

---

## Objetivos

Este notebook complementa el laboratorio principal de distribuciones, incluyendo:

- Distribuciones discretas adicionales: Binomial Negativa, Hipergeométrica, Multinomial
- Distribuciones continuas adicionales: Log-Normal, Weibull, Chi-cuadrado, t de Student, F de Fisher
- Exploración de propiedades especiales y aplicaciones

## Contenidos

- **Parte I:** Distribuciones Discretas Adicionales
- **Parte II:** Distribuciones Continuas Adicionales
- **Parte III:** Distribuciones para Inferencia Estadística

In [ ]:
# Librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.special import comb
import seaborn as sns
from ipywidgets import interact, FloatSlider, IntSlider
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10

# Configuración para reproducibilidad
np.random.seed(42)

---

# Parte I: Distribuciones Discretas Adicionales

## 1.1 Distribución Binomial Negativa

**Definición:** Número de fracasos antes de obtener $r$ éxitos en ensayos de Bernoulli independientes.

$$X \sim \text{BinomialNegativa}(r, p)$$

- **Soporte:** $\{0, 1, 2, \ldots\}$
- **Parámetros:** $r > 0$ (número de éxitos deseados), $p \in (0,1]$ (probabilidad de éxito)
- **PMF:** $P(X = k) = \binom{k+r-1}{k} p^r (1-p)^k$
- **Media:** $E[X] = \frac{r(1-p)}{p}$
- **Varianza:** $\text{Var}(X) = \frac{r(1-p)}{p^2}$

**Interpretación alternativa:** Si $Y$ = número de ensayos hasta obtener $r$ éxitos, entonces $Y = X + r$ y $Y \sim \text{BinomialNegativa}(r, p)$ (parametrización alternativa).

**Caso especial:** Cuando $r=1$, se reduce a la distribución Geométrica.

**Aplicaciones:** Modelado de conteos con sobredispersión (varianza > media), análisis de fallas, tiempo hasta eventos recurrentes.

In [ ]:
def plot_negative_binomial(r=5, p=0.3, n_sim=1000):
    """
    Visualiza la distribución Binomial Negativa y simulaciones.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PMF teórica
    x = np.arange(0, int(r*(1-p)/p * 3 + 20))
    pmf = stats.nbinom.pmf(x, r, p)
    axes[0].bar(x, pmf, color='teal', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Número de fracasos antes de r éxitos')
    axes[0].set_ylabel('Probabilidad')
    axes[0].set_title(f'PMF: Binomial Negativa(r={r}, p={p})')
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf = stats.nbinom.cdf(x, r, p)
    axes[1].step(x, cdf, where='post', color='darkblue', linewidth=2)
    axes[1].scatter(x, cdf, color='darkblue', s=20, zorder=3)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: Binomial Negativa(r={r}, p={p})')
    axes[1].grid(True, alpha=0.3)
    
    # Simulación
    samples = np.random.negative_binomial(r, p, n_sim)
    max_val = min(max(samples), int(r*(1-p)/p * 3 + 30))
    axes[2].hist(samples, bins=np.arange(-0.5, max_val+1.5, 1), density=True,
                 color='lightseagreen', alpha=0.7, edgecolor='black', label='Simulado')
    x_plot = x[x <= max_val]
    axes[2].plot(x_plot, stats.nbinom.pmf(x_plot, r, p), 'ro--', 
                 linewidth=2, markersize=6, label='Teórico')
    axes[2].set_xlabel('Número de fracasos')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print(f"Media teórica: {r*(1-p)/p:.4f}")
    print(f"Media empírica: {np.mean(samples):.4f}")
    print(f"Varianza teórica: {r*(1-p)/p**2:.4f}")
    print(f"Varianza empírica: {np.var(samples, ddof=1):.4f}")
    print(f"Razón Var/Media empírica: {np.var(samples, ddof=1)/np.mean(samples):.4f} (>1 indica sobredispersión)")

# Ejemplo
plot_negative_binomial(r=5, p=0.4, n_sim=1000)

### Exploración interactiva: Binomial Negativa

In [ ]:
interact(plot_negative_binomial, 
         r=IntSlider(min=1, max=20, step=1, value=5, description='r'),
         p=FloatSlider(min=0.1, max=0.9, step=0.1, value=0.3, description='p'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

## 1.2 Distribución Hipergeométrica

**Definición:** Número de éxitos en $n$ extracciones sin reemplazo de una población finita con $N$ elementos, de los cuales $K$ son éxitos.

$$X \sim \text{Hipergeométrica}(N, K, n)$$

- **Soporte:** $\{\max(0, n-(N-K)), \ldots, \min(n, K)\}$
- **Parámetros:** $N$ (tamaño población), $K$ (éxitos en población), $n$ (tamaño muestra)
- **PMF:** $P(X = k) = \frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}$
- **Media:** $E[X] = n\frac{K}{N}$
- **Varianza:** $\text{Var}(X) = n\frac{K}{N}\frac{N-K}{N}\frac{N-n}{N-1}$

**Diferencia con Binomial:** Hipergeométrica es para muestreo sin reemplazo (dependencia entre extracciones), Binomial es con reemplazo (independencia).

**Aplicaciones:** Control de calidad (lotes), pruebas de Fisher para tablas de contingencia, muestreo de poblaciones finitas.

In [ ]:
def plot_hypergeometric(N=50, K=20, n=10, n_sim=1000):
    """
    Visualiza la distribución Hipergeométrica y simulaciones.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PMF teórica
    x = np.arange(max(0, n-(N-K)), min(n, K)+1)
    pmf = stats.hypergeom.pmf(x, N, K, n)
    axes[0].bar(x, pmf, color='indianred', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Número de éxitos en la muestra')
    axes[0].set_ylabel('Probabilidad')
    axes[0].set_title(f'PMF: Hipergeométrica(N={N}, K={K}, n={n})')
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf = stats.hypergeom.cdf(x, N, K, n)
    axes[1].step(x, cdf, where='post', color='darkred', linewidth=2)
    axes[1].scatter(x, cdf, color='darkred', s=30, zorder=3)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: Hipergeométrica(N={N}, K={K}, n={n})')
    axes[1].grid(True, alpha=0.3)
    
    # Simulación
    samples = np.random.hypergeometric(K, N-K, n, n_sim)
    axes[2].hist(samples, bins=np.arange(min(x)-0.5, max(x)+1.5, 1), density=True,
                 color='lightcoral', alpha=0.7, edgecolor='black', label='Simulado')
    axes[2].plot(x, pmf, 'bo--', linewidth=2, markersize=6, label='Teórico')
    axes[2].set_xlabel('Número de éxitos')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    p = K/N
    print(f"Media teórica: {n*p:.4f}")
    print(f"Media empírica: {np.mean(samples):.4f}")
    var_teo = n * p * (1-p) * (N-n)/(N-1)
    print(f"Varianza teórica: {var_teo:.4f}")
    print(f"Varianza empírica: {np.var(samples, ddof=1):.4f}")
    print(f"\nFactor de corrección por población finita: (N-n)/(N-1) = {(N-n)/(N-1):.4f}")

# Ejemplo: Lote de 50 piezas, 20 defectuosas, extraemos 10
plot_hypergeometric(N=50, K=20, n=10, n_sim=1000)

### Exploración interactiva: Hipergeométrica

In [ ]:
interact(plot_hypergeometric, 
         N=IntSlider(min=20, max=100, step=10, value=50, description='N (población)'),
         K=IntSlider(min=5, max=50, step=5, value=20, description='K (éxitos)'),
         n=IntSlider(min=5, max=30, step=5, value=10, description='n (muestra)'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Comparación Hipergeométrica vs Binomial

Cuando $N$ es grande comparado con $n$ (población grande, muestra pequeña), la Hipergeométrica converge a la Binomial.

In [ ]:
def compare_hypergeom_binomial(K_prop=0.3, n=10):
    """
    Compara Hipergeométrica con Binomial para diferentes tamaños de población.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    N_values = [30, 100, 500, 5000]
    
    for idx, N in enumerate(N_values):
        K = int(N * K_prop)
        p = K / N
        
        x = np.arange(0, n+1)
        pmf_hyper = stats.hypergeom.pmf(x, N, K, n)
        pmf_binom = stats.binom.pmf(x, n, p)
        
        axes[idx].bar(x, pmf_hyper, alpha=0.6, label=f'Hipergeom(N={N}, K={K}, n={n})', color='coral')
        axes[idx].plot(x, pmf_binom, 'bo-', linewidth=2, markersize=5, label=f'Binomial(n={n}, p={p:.2f})')
        axes[idx].set_xlabel('k')
        axes[idx].set_ylabel('Probabilidad')
        axes[idx].set_title(f'N={N}, n/N={n/N:.3f}')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print(f"A medida que N aumenta (y n/N → 0), la Hipergeométrica converge a la Binomial.")

compare_hypergeom_binomial(K_prop=0.3, n=10)

## 1.3 Distribución Multinomial

**Definición:** Generalización de la Binomial para más de dos categorías. Cuenta el número de ocurrencias de cada categoría en $n$ ensayos independientes.

$$\mathbf{X} = (X_1, \ldots, X_k) \sim \text{Multinomial}(n, \mathbf{p})$$

donde $\mathbf{p} = (p_1, \ldots, p_k)$ con $\sum_{i=1}^k p_i = 1$

- **Soporte:** $\{(x_1, \ldots, x_k): x_i \geq 0, \sum x_i = n\}$
- **Parámetros:** $n$ (número de ensayos), $\mathbf{p}$ (probabilidades de cada categoría)
- **PMF:** $P(X_1=x_1, \ldots, X_k=x_k) = \frac{n!}{x_1! \cdots x_k!} p_1^{x_1} \cdots p_k^{x_k}$
- **Media:** $E[X_i] = np_i$
- **Varianza:** $\text{Var}(X_i) = np_i(1-p_i)$
- **Covarianza:** $\text{Cov}(X_i, X_j) = -np_ip_j$ para $i \neq j$

**Aplicaciones:** Genética (frecuencias alélicas), análisis de textos (bag of words), clasificación multiclase, pruebas chi-cuadrado de bondad de ajuste.

In [ ]:
def plot_multinomial(n=100, probs=[0.3, 0.5, 0.2], n_sim=1000):
    """
    Visualiza la distribución Multinomial mediante simulación.
    """
    k = len(probs)
    categories = [f'Cat {i+1}' for i in range(k)]
    
    # Simulación
    samples = np.random.multinomial(n, probs, n_sim)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Distribución marginal de cada categoría
    for i in range(k):
        axes[0].hist(samples[:, i], bins=30, alpha=0.5, label=categories[i], edgecolor='black')
    axes[0].set_xlabel('Conteo')
    axes[0].set_ylabel('Frecuencia')
    axes[0].set_title(f'Distribuciones Marginales\n(n={n}, p={probs})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Medias empíricas vs teóricas
    means_emp = np.mean(samples, axis=0)
    means_teo = np.array(probs) * n
    x_pos = np.arange(k)
    width = 0.35
    axes[1].bar(x_pos - width/2, means_teo, width, label='Teórico', alpha=0.7, color='steelblue')
    axes[1].bar(x_pos + width/2, means_emp, width, label='Empírico', alpha=0.7, color='coral')
    axes[1].set_xlabel('Categoría')
    axes[1].set_ylabel('Media')
    axes[1].set_title('Medias: Teóricas vs Empíricas')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(categories)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Matriz de covarianzas empírica
    cov_emp = np.cov(samples.T)
    im = axes[2].imshow(cov_emp, cmap='coolwarm', aspect='auto')
    axes[2].set_xticks(range(k))
    axes[2].set_yticks(range(k))
    axes[2].set_xticklabels(categories)
    axes[2].set_yticklabels(categories)
    axes[2].set_title('Matriz de Covarianzas Empírica')
    plt.colorbar(im, ax=axes[2])
    
    # Añadir valores en la matriz
    for i in range(k):
        for j in range(k):
            text = axes[2].text(j, i, f'{cov_emp[i, j]:.1f}',
                               ha="center", va="center", color="black", fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print("Medias teóricas:", means_teo)
    print("Medias empíricas:", means_emp)
    print("\nVarianzas teóricas:", n * np.array(probs) * (1 - np.array(probs)))
    print("Varianzas empíricas:", np.var(samples, axis=0, ddof=1))
    print("\nNota: Las covarianzas son negativas (competencia por el total n)")

# Ejemplo: tirar un dado cargado 100 veces
plot_multinomial(n=100, probs=[0.15, 0.15, 0.15, 0.15, 0.15, 0.25], n_sim=1000)

### Ejemplo aplicado: Genética (Ley de Hardy-Weinberg)

En una población con dos alelos A y a, los genotipos AA, Aa, aa tienen frecuencias teóricas $p^2, 2pq, q^2$ donde $p+q=1$.

In [ ]:
def hardy_weinberg_simulation(p=0.6, n_individuals=200, n_sim=1000):
    """
    Simula genotipos bajo equilibrio de Hardy-Weinberg.
    """
    q = 1 - p
    # Frecuencias teóricas de AA, Aa, aa
    probs_hw = [p**2, 2*p*q, q**2]
    
    samples = np.random.multinomial(n_individuals, probs_hw, n_sim)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    genotypes = ['AA', 'Aa', 'aa']
    colors = ['darkblue', 'purple', 'darkred']
    
    # Histogramas de cada genotipo
    for i, (geno, color) in enumerate(zip(genotypes, colors)):
        axes[0].hist(samples[:, i], bins=30, alpha=0.6, label=geno, 
                     color=color, edgecolor='black')
    axes[0].axvline(n_individuals * probs_hw[0], color='darkblue', linestyle='--', linewidth=2)
    axes[0].axvline(n_individuals * probs_hw[1], color='purple', linestyle='--', linewidth=2)
    axes[0].axvline(n_individuals * probs_hw[2], color='darkred', linestyle='--', linewidth=2)
    axes[0].set_xlabel('Conteo de individuos')
    axes[0].set_ylabel('Frecuencia')
    axes[0].set_title(f'Genotipos bajo Hardy-Weinberg\n(p={p}, q={q}, n={n_individuals})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Comparación medias
    means_emp = np.mean(samples, axis=0)
    means_teo = np.array(probs_hw) * n_individuals
    x_pos = np.arange(3)
    width = 0.35
    axes[1].bar(x_pos - width/2, means_teo, width, label='Teórico (H-W)', 
                alpha=0.7, color='steelblue')
    axes[1].bar(x_pos + width/2, means_emp, width, label='Empírico', 
                alpha=0.7, color='coral')
    axes[1].set_xlabel('Genotipo')
    axes[1].set_ylabel('Número promedio de individuos')
    axes[1].set_title('Equilibrio Hardy-Weinberg')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(genotypes)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Frecuencias teóricas (H-W): AA={probs_hw[0]:.4f}, Aa={probs_hw[1]:.4f}, aa={probs_hw[2]:.4f}")
    print(f"Conteos esperados: AA={means_teo[0]:.2f}, Aa={means_teo[1]:.2f}, aa={means_teo[2]:.2f}")
    print(f"Conteos empíricos: AA={means_emp[0]:.2f}, Aa={means_emp[1]:.2f}, aa={means_emp[2]:.2f}")

hardy_weinberg_simulation(p=0.6, n_individuals=200, n_sim=1000)

---

# Parte II: Distribuciones Continuas Adicionales

## 2.1 Distribución Log-Normal

**Definición:** Si $\ln(X) \sim \mathcal{N}(\mu, \sigma^2)$, entonces $X \sim \text{Log-Normal}(\mu, \sigma^2)$.

$$X \sim \text{Log-Normal}(\mu, \sigma^2)$$

- **Soporte:** $(0, \infty)$
- **Parámetros:** $\mu \in \mathbb{R}$ (media del logaritmo), $\sigma^2 > 0$ (varianza del logaritmo)
- **PDF:** $f(x) = \frac{1}{x\sigma\sqrt{2\pi}} \exp\left(-\frac{(\ln x - \mu)^2}{2\sigma^2}\right)$ para $x > 0$
- **Media:** $E[X] = e^{\mu + \sigma^2/2}$
- **Varianza:** $\text{Var}(X) = (e^{\sigma^2} - 1)e^{2\mu + \sigma^2}$
- **Mediana:** $e^\mu$
- **Moda:** $e^{\mu - \sigma^2}$

**Propiedades:**
- Asimétrica positiva (cola derecha pesada)
- Producto de variables Log-Normales es Log-Normal
- Media > Mediana > Moda

**Aplicaciones:** Precios de acciones, ingresos, tamaños de partículas, tiempos de vida útil, concentraciones de contaminantes.

In [ ]:
def plot_lognormal(mu=0, sigma=1, n_sim=1000):
    """
    Visualiza la distribución Log-Normal y simulaciones.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PDF teórica
    x = np.linspace(0.01, np.exp(mu + 3*sigma), 500)
    pdf = stats.lognorm.pdf(x, s=sigma, scale=np.exp(mu))
    axes[0].plot(x, pdf, 'darkorange', linewidth=2, label='PDF')
    axes[0].fill_between(x, pdf, alpha=0.3, color='orange')
    
    # Marcar estadísticos
    mean_val = np.exp(mu + sigma**2/2)
    median_val = np.exp(mu)
    mode_val = np.exp(mu - sigma**2)
    
    axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Media={mean_val:.2f}')
    axes[0].axvline(median_val, color='green', linestyle='--', linewidth=1.5, label=f'Mediana={median_val:.2f}')
    axes[0].axvline(mode_val, color='blue', linestyle='--', linewidth=1.5, label=f'Moda={mode_val:.2f}')
    
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: Log-Normal(μ={mu}, σ²={sigma**2})')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf = stats.lognorm.cdf(x, s=sigma, scale=np.exp(mu))
    axes[1].plot(x, cdf, 'darkorange', linewidth=2, label='FDA')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: Log-Normal(μ={mu}, σ²={sigma**2})')
    axes[1].set_ylim([-0.1, 1.1])
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    # Simulación
    samples = np.random.lognormal(mu, sigma, n_sim)
    axes[2].hist(samples, bins=50, density=True, color='wheat', 
                 alpha=0.7, edgecolor='black', label='Simulado')
    axes[2].plot(x, pdf, 'r-', linewidth=2, label='Teórico')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].set_xlim([0, np.percentile(samples, 99)])
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    var_teo = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)
    print(f"Media teórica: {mean_val:.4f}")
    print(f"Media empírica: {np.mean(samples):.4f}")
    print(f"Mediana teórica: {median_val:.4f}")
    print(f"Mediana empírica: {np.median(samples):.4f}")
    print(f"Varianza teórica: {var_teo:.4f}")
    print(f"Varianza empírica: {np.var(samples, ddof=1):.4f}")

# Ejemplo
plot_lognormal(mu=1, sigma=0.7, n_sim=1000)

### Exploración interactiva: Log-Normal

In [ ]:
interact(plot_lognormal, 
         mu=FloatSlider(min=-2, max=2, step=0.5, value=0, description='μ'),
         sigma=FloatSlider(min=0.3, max=2, step=0.1, value=1, description='σ'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Relación con la Normal

Si $X \sim \text{Log-Normal}(\mu, \sigma^2)$, entonces $\ln(X) \sim \mathcal{N}(\mu, \sigma^2)$.

In [ ]:
def show_lognormal_transform(mu=0, sigma=1, n_sim=5000):
    """
    Muestra la relación entre Log-Normal y Normal mediante transformación logarítmica.
    """
    samples_lognormal = np.random.lognormal(mu, sigma, n_sim)
    samples_log_transformed = np.log(samples_lognormal)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Log-Normal
    axes[0].hist(samples_lognormal, bins=50, density=True, color='wheat', 
                 alpha=0.7, edgecolor='black', label='Datos originales')
    x = np.linspace(0.01, np.percentile(samples_lognormal, 99), 500)
    pdf_lognorm = stats.lognorm.pdf(x, s=sigma, scale=np.exp(mu))
    axes[0].plot(x, pdf_lognorm, 'r-', linewidth=2, label='Log-Normal teórica')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Densidad')
    axes[0].set_title(f'X ~ Log-Normal(μ={mu}, σ²={sigma**2})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Después de ln(X)
    axes[1].hist(samples_log_transformed, bins=50, density=True, color='skyblue', 
                 alpha=0.7, edgecolor='black', label='ln(X)')
    x_norm = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
    pdf_norm = stats.norm.pdf(x_norm, loc=mu, scale=sigma)
    axes[1].plot(x_norm, pdf_norm, 'r-', linewidth=2, label='Normal teórica')
    axes[1].set_xlabel('ln(X)')
    axes[1].set_ylabel('Densidad')
    axes[1].set_title(f'ln(X) ~ Normal(μ={mu}, σ²={sigma**2})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("La transformación logarítmica convierte una Log-Normal en Normal.")
    print(f"Media de ln(X): teórica={mu:.4f}, empírica={np.mean(samples_log_transformed):.4f}")
    print(f"Desv.Est de ln(X): teórica={sigma:.4f}, empírica={np.std(samples_log_transformed, ddof=1):.4f}")

show_lognormal_transform(mu=1, sigma=0.8, n_sim=5000)

## 2.2 Distribución Weibull

**Definición:** Distribución de tiempos de vida con tasa de falla no constante (puede aumentar, disminuir o ser constante).

$$X \sim \text{Weibull}(k, \lambda)$$

- **Soporte:** $[0, \infty)$
- **Parámetros:** $k > 0$ (forma/shape), $\lambda > 0$ (escala/scale)
- **PDF:** $f(x) = \frac{k}{\lambda}\left(\frac{x}{\lambda}\right)^{k-1} e^{-(x/\lambda)^k}$ para $x \geq 0$
- **FDA:** $F(x) = 1 - e^{-(x/\lambda)^k}$ para $x \geq 0$
- **Media:** $E[X] = \lambda \Gamma(1 + 1/k)$
- **Varianza:** $\text{Var}(X) = \lambda^2 \left[\Gamma(1 + 2/k) - \Gamma^2(1 + 1/k)\right]$

**Interpretación del parámetro $k$:**
- $k < 1$: tasa de falla decreciente (mortalidad infantil)
- $k = 1$: tasa de falla constante (Exponencial)
- $k > 1$: tasa de falla creciente (envejecimiento)

**Aplicaciones:** Análisis de supervivencia, tiempos de falla de componentes, confiabilidad de sistemas, meteorología (vientos).

In [ ]:
def plot_weibull(k=1.5, lam=1, n_sim=1000):
    """
    Visualiza la distribución Weibull y simulaciones.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PDF teórica
    x = np.linspace(0.01, lam * 4, 500)
    pdf = stats.weibull_min.pdf(x, c=k, scale=lam)
    axes[0].plot(x, pdf, 'darkgreen', linewidth=2, label='PDF')
    axes[0].fill_between(x, pdf, alpha=0.3, color='green')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: Weibull(k={k}, λ={lam})')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # FDA teórica
    cdf = stats.weibull_min.cdf(x, c=k, scale=lam)
    axes[1].plot(x, cdf, 'darkgreen', linewidth=2, label='FDA')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: Weibull(k={k}, λ={lam})')
    axes[1].set_ylim([-0.1, 1.1])
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    # Simulación
    samples = np.random.weibull(k, n_sim) * lam
    axes[2].hist(samples, bins=40, density=True, color='lightgreen', 
                 alpha=0.7, edgecolor='black', label='Simulado')
    axes[2].plot(x, pdf, 'r-', linewidth=2, label='Teórico')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].set_xlim([0, np.percentile(samples, 99)])
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    from scipy.special import gamma
    mean_teo = lam * gamma(1 + 1/k)
    var_teo = lam**2 * (gamma(1 + 2/k) - gamma(1 + 1/k)**2)
    
    print(f"Media teórica: {mean_teo:.4f}")
    print(f"Media empírica: {np.mean(samples):.4f}")
    print(f"Varianza teórica: {var_teo:.4f}")
    print(f"Varianza empírica: {np.var(samples, ddof=1):.4f}")
    
    if k < 1:
        print(f"\nk={k} < 1: Tasa de falla decreciente (mortalidad infantil)")
    elif k == 1:
        print(f"\nk={k} = 1: Tasa de falla constante (equivalente a Exponencial)")
    else:
        print(f"\nk={k} > 1: Tasa de falla creciente (envejecimiento)")

# Ejemplo
plot_weibull(k=2, lam=1.5, n_sim=1000)

### Exploración interactiva: Weibull

In [ ]:
interact(plot_weibull, 
         k=FloatSlider(min=0.5, max=5, step=0.5, value=1.5, description='k (forma)'),
         lam=FloatSlider(min=0.5, max=3, step=0.5, value=1, description='λ (escala)'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Comparación de diferentes valores de k

In [ ]:
def compare_weibull_shapes(lam=1):
    """
    Compara la forma de la Weibull para diferentes valores de k.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(0.01, 4, 500)
    k_values = [0.5, 1, 1.5, 2, 3]
    colors = plt.cm.viridis(np.linspace(0, 1, len(k_values)))
    
    # PDF
    for k, color in zip(k_values, colors):
        pdf = stats.weibull_min.pdf(x, c=k, scale=lam)
        label = f'k={k}'
        if k < 1:
            label += ' (decrec.)'
        elif k == 1:
            label += ' (const.)'
        else:
            label += ' (crec.)'
        axes[0].plot(x, pdf, linewidth=2, label=label, color=color)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: Weibull con λ={lam}, variando k')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Función de tasa de falla h(x) = f(x)/(1-F(x))
    for k, color in zip(k_values, colors):
        pdf = stats.weibull_min.pdf(x, c=k, scale=lam)
        cdf = stats.weibull_min.cdf(x, c=k, scale=lam)
        hazard = pdf / (1 - cdf + 1e-10)  # evitar división por cero
        axes[1].plot(x, hazard, linewidth=2, label=f'k={k}', color=color)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('h(x) - Tasa de falla')
    axes[1].set_title('Función de tasa de falla (hazard rate)')
    axes[1].set_ylim([0, 3])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Interpretación de la tasa de falla:")
    print("- k < 1: La tasa de falla disminuye con el tiempo (los que sobreviven son más resistentes)")
    print("- k = 1: Tasa de falla constante (fallas aleatorias, sin memoria)")
    print("- k > 1: La tasa de falla aumenta con el tiempo (envejecimiento, desgaste)")

compare_weibull_shapes(lam=1)

---

# Parte III: Distribuciones para Inferencia Estadística

## 3.1 Distribución Chi-cuadrado

**Definición:** Suma de cuadrados de $n$ variables Normales estándar independientes. Caso especial de la Gamma.

$$X \sim \chi^2_n$$

Si $Z_1, \ldots, Z_n \sim \mathcal{N}(0,1)$ independientes, entonces $X = \sum_{i=1}^n Z_i^2 \sim \chi^2_n$

- **Soporte:** $[0, \infty)$
- **Parámetro:** $n$ (grados de libertad)
- **PDF:** $f(x) = \frac{1}{2^{n/2}\Gamma(n/2)} x^{n/2-1} e^{-x/2}$ para $x > 0$
- **Media:** $E[X] = n$
- **Varianza:** $\text{Var}(X) = 2n$

**Relación con otras distribuciones:**
- $\chi^2_n = \text{Gamma}(n/2, 1/2)$
- Si $X \sim \chi^2_n$ e $Y \sim \chi^2_m$ son independientes, entonces $X+Y \sim \chi^2_{n+m}$

**Aplicaciones:** Pruebas de bondad de ajuste, pruebas de independencia, intervalos de confianza para varianza, estimación de varianzas.

In [ ]:
def plot_chisquare(df=5, n_sim=1000):
    """
    Visualiza la distribución Chi-cuadrado y su construcción desde Normales.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PDF teórica
    x = np.linspace(0.01, df + 4*np.sqrt(2*df), 500)
    pdf = stats.chi2.pdf(x, df)
    axes[0].plot(x, pdf, 'crimson', linewidth=2, label='PDF')
    axes[0].fill_between(x, pdf, alpha=0.3, color='red')
    axes[0].axvline(df, color='blue', linestyle='--', linewidth=1.5, label=f'Media = {df}')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: χ²({df} grados de libertad)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf = stats.chi2.cdf(x, df)
    axes[1].plot(x, cdf, 'crimson', linewidth=2, label='FDA')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: χ²({df})')
    axes[1].set_ylim([-0.1, 1.1])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Simulación: construir desde Normales
    normals = np.random.normal(0, 1, (n_sim, df))
    samples_constructed = np.sum(normals**2, axis=1)
    samples_direct = np.random.chisquare(df, n_sim)
    
    axes[2].hist(samples_constructed, bins=40, density=True, color='lightcoral', 
                 alpha=0.7, edgecolor='black', label='Suma de Z²')
    axes[2].plot(x, pdf, 'b-', linewidth=2, label='Teórico')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación: Σ(Z_i²), n_sim={n_sim}')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print(f"Media teórica: {df:.4f}")
    print(f"Media empírica (construcción): {np.mean(samples_constructed):.4f}")
    print(f"Media empírica (directa): {np.mean(samples_direct):.4f}")
    print(f"Varianza teórica: {2*df:.4f}")
    print(f"Varianza empírica (construcción): {np.var(samples_constructed, ddof=1):.4f}")

# Ejemplo
plot_chisquare(df=7, n_sim=1000)

### Exploración interactiva: Chi-cuadrado

In [ ]:
interact(plot_chisquare, 
         df=IntSlider(min=1, max=30, step=1, value=5, description='df (gl)'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Efecto de los grados de libertad

In [ ]:
def compare_chisquare_df():
    """
    Compara Chi-cuadrado para diferentes grados de libertad.
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    
    df_values = [1, 2, 3, 5, 10, 20]
    colors = plt.cm.Reds(np.linspace(0.3, 1, len(df_values)))
    
    for df, color in zip(df_values, colors):
        x = np.linspace(0.01, df + 4*np.sqrt(2*df), 500)
        pdf = stats.chi2.pdf(x, df)
        ax.plot(x, pdf, linewidth=2, label=f'df={df}', color=color)
    
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_title('Distribución χ² para diferentes grados de libertad')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 40])
    
    plt.tight_layout()
    plt.show()
    
    print("A medida que los grados de libertad aumentan:")
    print("- La distribución se desplaza hacia la derecha")
    print("- Se vuelve más simétrica y se aproxima a una Normal")
    print("- Por el TCL: χ²(n) → N(n, 2n) cuando n → ∞")

compare_chisquare_df()

## 3.2 Distribución t de Student

**Definición:** Razón entre una Normal estándar y la raíz cuadrada de una Chi-cuadrado dividida por sus grados de libertad.

$$T = \frac{Z}{\sqrt{V/n}} \sim t_n$$

donde $Z \sim \mathcal{N}(0,1)$ y $V \sim \chi^2_n$ son independientes.

- **Soporte:** $(-\infty, \infty)$
- **Parámetro:** $n$ (grados de libertad)
- **PDF:** $f(x) = \frac{\Gamma((n+1)/2)}{\sqrt{n\pi}\Gamma(n/2)} \left(1 + \frac{x^2}{n}\right)^{-(n+1)/2}$
- **Media:** $E[T] = 0$ para $n > 1$
- **Varianza:** $\text{Var}(T) = \frac{n}{n-2}$ para $n > 2$

**Propiedades:**
- Simétrica respecto a 0
- Colas más pesadas que la Normal
- Cuando $n \to \infty$, $t_n \to \mathcal{N}(0,1)$

**Aplicaciones:** Intervalos de confianza para la media con varianza desconocida, pruebas t, regresión lineal (inferencia sobre coeficientes).

In [ ]:
def plot_t_student(df=5, n_sim=1000):
    """
    Visualiza la distribución t de Student y su construcción.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PDF teórica vs Normal
    x = np.linspace(-5, 5, 500)
    pdf_t = stats.t.pdf(x, df)
    pdf_norm = stats.norm.pdf(x, 0, 1)
    
    axes[0].plot(x, pdf_t, 'darkblue', linewidth=2, label=f't({df} gl)')
    axes[0].plot(x, pdf_norm, 'r--', linewidth=2, label='N(0,1)')
    axes[0].fill_between(x, pdf_t, alpha=0.3, color='blue')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: t de Student vs Normal')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf_t = stats.t.cdf(x, df)
    axes[1].plot(x, cdf_t, 'darkblue', linewidth=2, label='FDA')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: t({df})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Simulación: construir como Z/sqrt(V/n)
    Z = np.random.normal(0, 1, n_sim)
    V = np.random.chisquare(df, n_sim)
    samples_constructed = Z / np.sqrt(V / df)
    samples_direct = np.random.standard_t(df, n_sim)
    
    # Filtrar outliers extremos para mejor visualización
    samples_plot = samples_constructed[(samples_constructed > -10) & (samples_constructed < 10)]
    
    axes[2].hist(samples_plot, bins=50, density=True, color='lightblue', 
                 alpha=0.7, edgecolor='black', label='Z/√(V/n)')
    axes[2].plot(x, pdf_t, 'r-', linewidth=2, label='Teórico')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].set_xlim([-5, 5])
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print(f"Media teórica: 0.0000")
    print(f"Media empírica: {np.mean(samples_constructed):.4f}")
    if df > 2:
        print(f"Varianza teórica: {df/(df-2):.4f}")
        print(f"Varianza empírica: {np.var(samples_constructed, ddof=1):.4f}")
    else:
        print(f"Varianza: indefinida para df ≤ 2")
    print(f"\nColas más pesadas que N(0,1): útil para muestras pequeñas.")

# Ejemplo
plot_t_student(df=5, n_sim=1000)

### Exploración interactiva: t de Student

In [ ]:
interact(plot_t_student, 
         df=IntSlider(min=1, max=30, step=1, value=5, description='df (gl)'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Convergencia de t a Normal

In [ ]:
def show_t_convergence():
    """
    Muestra cómo t de Student converge a Normal cuando df → ∞.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    df_values = [1, 3, 10, 30]
    x = np.linspace(-4, 4, 500)
    pdf_norm = stats.norm.pdf(x, 0, 1)
    
    for idx, df in enumerate(df_values):
        pdf_t = stats.t.pdf(x, df)
        
        axes[idx].plot(x, pdf_t, 'b-', linewidth=2.5, label=f't({df} gl)')
        axes[idx].plot(x, pdf_norm, 'r--', linewidth=2, label='N(0,1)')
        axes[idx].fill_between(x, pdf_t, alpha=0.2, color='blue')
        axes[idx].set_xlabel('x')
        axes[idx].set_ylabel('f(x)')
        axes[idx].set_title(f'df = {df}')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
        axes[idx].set_ylim([0, 0.45])
    
    plt.tight_layout()
    plt.show()
    
    print("A medida que los grados de libertad aumentan:")
    print("- Las colas de la t de Student se hacen más ligeras")
    print("- La distribución converge a N(0,1)")
    print("- Para df ≥ 30, la diferencia es prácticamente despreciable")

show_t_convergence()

## 3.3 Distribución F de Fisher

**Definición:** Razón de dos variables Chi-cuadrado independientes, cada una dividida por sus grados de libertad.

$$F = \frac{U/d_1}{V/d_2} \sim F_{d_1, d_2}$$

donde $U \sim \chi^2_{d_1}$ y $V \sim \chi^2_{d_2}$ son independientes.

- **Soporte:** $[0, \infty)$
- **Parámetros:** $d_1, d_2$ (grados de libertad del numerador y denominador)
- **PDF:** Compleja (ver fórmula en textos especializados)
- **Media:** $E[F] = \frac{d_2}{d_2 - 2}$ para $d_2 > 2$
- **Varianza:** $\text{Var}(F) = \frac{2d_2^2(d_1+d_2-2)}{d_1(d_2-2)^2(d_2-4)}$ para $d_2 > 4$

**Propiedades:**
- Asimétrica positiva
- Si $T \sim t_n$, entonces $T^2 \sim F_{1,n}$
- Si $F \sim F_{d_1, d_2}$, entonces $1/F \sim F_{d_2, d_1}$

**Aplicaciones:** ANOVA (comparación de varianzas), pruebas F en regresión lineal, comparación de modelos.

In [ ]:
def plot_f_fisher(df1=5, df2=10, n_sim=1000):
    """
    Visualiza la distribución F de Fisher y su construcción.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # PDF teórica
    x = np.linspace(0.01, 5, 500)
    pdf = stats.f.pdf(x, df1, df2)
    axes[0].plot(x, pdf, 'darkviolet', linewidth=2, label='PDF')
    axes[0].fill_between(x, pdf, alpha=0.3, color='violet')
    if df2 > 2:
        mean_val = df2 / (df2 - 2)
        axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=1.5, 
                       label=f'Media={mean_val:.2f}')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'PDF: F({df1}, {df2})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # FDA teórica
    cdf = stats.f.cdf(x, df1, df2)
    axes[1].plot(x, cdf, 'darkviolet', linewidth=2, label='FDA')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('F(x)')
    axes[1].set_title(f'FDA: F({df1}, {df2})')
    axes[1].set_ylim([-0.1, 1.1])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Simulación: construir como (U/d1)/(V/d2)
    U = np.random.chisquare(df1, n_sim)
    V = np.random.chisquare(df2, n_sim)
    samples_constructed = (U / df1) / (V / df2)
    samples_direct = np.random.f(df1, df2, n_sim)
    
    # Filtrar outliers extremos
    samples_plot = samples_constructed[samples_constructed < 10]
    
    axes[2].hist(samples_plot, bins=50, density=True, color='plum', 
                 alpha=0.7, edgecolor='black', label='(U/d₁)/(V/d₂)')
    axes[2].plot(x, pdf, 'r-', linewidth=2, label='Teórico')
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Densidad')
    axes[2].set_title(f'Simulación (n_sim={n_sim})')
    axes[2].set_xlim([0, 5])
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    if df2 > 2:
        mean_teo = df2 / (df2 - 2)
        print(f"Media teórica: {mean_teo:.4f}")
        print(f"Media empírica: {np.mean(samples_constructed):.4f}")
    else:
        print("Media: indefinida para df2 ≤ 2")
    
    if df2 > 4:
        var_teo = (2 * df2**2 * (df1 + df2 - 2)) / (df1 * (df2-2)**2 * (df2-4))
        print(f"Varianza teórica: {var_teo:.4f}")
        print(f"Varianza empírica: {np.var(samples_constructed, ddof=1):.4f}")
    else:
        print("Varianza: indefinida para df2 ≤ 4")

# Ejemplo
plot_f_fisher(df1=5, df2=10, n_sim=1000)

### Exploración interactiva: F de Fisher

In [ ]:
interact(plot_f_fisher, 
         df1=IntSlider(min=1, max=20, step=1, value=5, description='df1 (numer)'),
         df2=IntSlider(min=3, max=30, step=1, value=10, description='df2 (denom)'),
         n_sim=IntSlider(min=500, max=5000, step=500, value=1000, description='n_sim'));

### Efecto de los grados de libertad en F

In [ ]:
def compare_f_distributions():
    """
    Compara distribuciones F con diferentes grados de libertad.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.linspace(0.01, 5, 500)
    
    # Variando df1 (numerador)
    df2_fixed = 10
    for df1 in [1, 3, 5, 10, 20]:
        pdf = stats.f.pdf(x, df1, df2_fixed)
        axes[0].plot(x, pdf, linewidth=2, label=f'F({df1}, {df2_fixed})')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title(f'Variando df₁ (numerador), df₂={df2_fixed} fijo')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Variando df2 (denominador)
    df1_fixed = 5
    for df2 in [3, 5, 10, 20, 50]:
        pdf = stats.f.pdf(x, df1_fixed, df2)
        axes[1].plot(x, pdf, linewidth=2, label=f'F({df1_fixed}, {df2})')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('f(x)')
    axes[1].set_title(f'Variando df₂ (denominador), df₁={df1_fixed} fijo')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Observaciones:")
    print("- A mayor df₁, la distribución se ensancha")
    print("- A mayor df₂, la distribución se concentra más cerca de 1")
    print("- Cuando ambos df → ∞, F → 1")

compare_f_distributions()

---

## Resumen de Distribuciones

### Discretas

| Distribución | Parámetros | Media | Aplicación típica |
|--------------|------------|-------|-------------------|
| Binomial Negativa | $r, p$ | $r(1-p)/p$ | Conteos con sobredispersión |
| Hipergeométrica | $N, K, n$ | $n(K/N)$ | Muestreo sin reemplazo |
| Multinomial | $n, \mathbf{p}$ | $np_i$ | Categorías múltiples |

### Continuas

| Distribución | Parámetros | Media | Aplicación típica |
|--------------|------------|-------|-------------------|
| Log-Normal | $\mu, \sigma^2$ | $e^{\mu+\sigma^2/2}$ | Precios, ingresos |
| Weibull | $k, \lambda$ | $\lambda\Gamma(1+1/k)$ | Análisis de supervivencia |
| Chi-cuadrado | $n$ | $n$ | Pruebas de bondad de ajuste |
| t de Student | $n$ | $0$ | Inferencia con $\sigma$ desconocida |
| F de Fisher | $d_1, d_2$ | $d_2/(d_2-2)$ | ANOVA, comparación de varianzas |

---

## Ejercicios Adicionales

### Ejercicio 1: Sobredispersión
Genera datos de una Binomial Negativa y de una Poisson con la misma media. Compara la razón Var/Media en ambos casos.

### Ejercicio 2: Muestreo sin reemplazo
Simula el problema clásico: urna con 10 bolas rojas y 15 azules. Extrae 5 sin reemplazo. ¿Cuántas rojas esperas? Compara con el caso con reemplazo.

### Ejercicio 3: Transformaciones
Si $X \sim \text{Log-Normal}(0, 1)$, verifica que $\ln(X) \sim \mathcal{N}(0, 1)$ mediante simulación.

### Ejercicio 4: Análisis de supervivencia
Simula tiempos de falla de componentes usando Weibull$(k=2.5, \lambda=100)$. Calcula la probabilidad de supervivencia más allá de 80 horas.

### Ejercicio 5: Inferencia
Genera una muestra de $n=15$ observaciones de $\mathcal{N}(10, 4)$. Construye un intervalo de confianza del 95% para la media usando la distribución t de Student.

---

**Material adicional:** Este notebook complementa las distribuciones vistas en clase. Para profundizar, consultar Casella & Berger, capítulos 3-4.